In [3]:
from queue import PriorityQueue

class Point:
    def __init__(self, location, previous=None):
        self.location = location
        self.previous = previous
        self.cost_from_start = 0
        self.heuristic_value = 0
        self.total_estimate = 0
    
    def __lt__(self, other):
        return self.total_estimate < other.total_estimate

def compute_heuristic(current_point, target_point):
    return abs(current_point[0] - target_point[0]) + abs(current_point[1] - target_point[1])

def greedy_search_multiple_targets(environment, origin, targets):
    row_count, col_count = len(environment), len(environment[0])
    remaining_targets = targets.copy()
    current_location = origin
    complete_route = [origin]
    overall_cost = 0
    
    while remaining_targets:
        closest_target = None
        smallest_distance = float('inf')
        
        for target in remaining_targets:
            distance = compute_heuristic(current_location, target)
            if distance < smallest_distance:
                smallest_distance = distance
                closest_target = target
        
        route_to_target = greedy_search_single_target(environment, current_location, closest_target)
        
        if route_to_target:
            if len(route_to_target) > 1:
                complete_route.extend(route_to_target[1:])
                overall_cost += len(route_to_target) - 1
            
            current_location = closest_target
            remaining_targets.remove(closest_target)
        else:
            return None, float('inf')
    
    return complete_route, overall_cost

def greedy_search_single_target(environment, origin, target):
    row_count, col_count = len(environment), len(environment[0])
    start_point = Point(origin)
    destination_point = Point(target)
    
    priority_queue = PriorityQueue()
    priority_queue.put(start_point)
    explored = set()
    
    while not priority_queue.empty():
        current_point = priority_queue.get()
        current_coords = current_point.location
        
        if current_coords == target:
            route = []
            while current_point:
                route.append(current_point.location)
                current_point = current_point.previous
            return route[::-1]
        
        explored.add(current_coords)
        
        for dx, dy in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            new_coords = (current_coords[0] + dx, current_coords[1] + dy)
            
            if (0 <= new_coords[0] < row_count and 
                0 <= new_coords[1] < col_count and 
                environment[new_coords[0]][new_coords[1]] == 0 and 
                new_coords not in explored):
                
                new_point = Point(new_coords, current_point)
                new_point.cost_from_start = current_point.cost_from_start + 1
                new_point.heuristic_value = compute_heuristic(new_coords, target)
                new_point.total_estimate = new_point.heuristic_value
                
                priority_queue.put(new_point)
                explored.add(new_coords)
    
    return None

environment = [
    [0, 0, 1, 0, 0, 0, 1, 0],
    [0, 1, 1, 0, 1, 0, 1, 0],
    [0, 0, 0, 0, 1, 0, 0, 0],
    [1, 1, 1, 0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 1, 0],
    [0, 1, 1, 1, 1, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 1, 0],
    [0, 1, 1, 0, 0, 0, 1, 0]
]

origin = (0, 0)
targets = [(2, 7), (5, 7), (7, 3), (4, 4)]

print("Multi-Goal Path Finding")
print(f"Starting Point: {origin}")
print(f"Destination Points: {targets}")

route, cost = greedy_search_multiple_targets(environment, origin, targets)

if route:
    print(f"Route covering all destinations: {route}")
    print(f"Total moves required: {cost}")
else:
    print("Unable to find a route to all destinations!")
print()

Multi-Goal Path Finding
Starting Point: (0, 0)
Destination Points: [(2, 7), (5, 7), (7, 3), (4, 4)]
Route covering all destinations: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (3, 3), (4, 3), (4, 4), (4, 5), (5, 5), (5, 6), (5, 7), (4, 7), (3, 7), (2, 7), (3, 7), (4, 7), (5, 7), (5, 6), (5, 5), (6, 5), (7, 5), (7, 4), (7, 3)]
Total moves required: 24



In [4]:
import random
import time
from queue import PriorityQueue

class DynamicAStar:
    def __init__(self, network, heuristic_values, original_weights):
        self.network = network
        self.heuristic_values = heuristic_values
        self.original_weights = original_weights
        self.current_weights = self.duplicate_weights(original_weights)
        self.previous_update = time.time()
        self.refresh_rate = 3
        
    def duplicate_weights(self, weights):
        new_weights = {}
        for point, connections in weights.items():
            new_weights[point] = connections.copy()
        return new_weights
    
    def randomly_adjust_weights(self):
        now = time.time()
        if now - self.previous_update >= self.refresh_rate:
            for point in self.current_weights:
                for neighbor in self.current_weights[point]:
                    adjustment = random.uniform(0.7, 1.5)
                    base_value = self.original_weights[point][neighbor]
                    updated_value = round(base_value * adjustment, 1)
                    self.current_weights[point][neighbor] = updated_value
            
            self.previous_update = now
            return True
        return False
    
    def find_path_astar(self, starting_point, target_point, max_attempts=100):
        priority_q = PriorityQueue()
        priority_q.put((self.heuristic_values[starting_point], starting_point, [starting_point]))
        travel_cost = {starting_point: 0}
        explored = set()
        attempts = 0
        
        while not priority_q.empty() and attempts < max_attempts:
            attempts += 1
            self.randomly_adjust_weights()
            
            estimated_cost, current_spot, route = priority_q.get()
            
            if current_spot in explored:
                continue
            
            spent_so_far = travel_cost[current_spot]
            
            if current_spot == target_point:
                return route, spent_so_far
            
            explored.add(current_spot)
            
            for next_spot, path_cost in self.current_weights[current_spot].items():
                if next_spot not in explored:
                    new_spent = spent_so_far + path_cost
                    
                    if next_spot not in travel_cost or new_spent < travel_cost[next_spot]:
                        travel_cost[next_spot] = new_spent
                        estimated_cost = new_spent + self.heuristic_values[next_spot]
                        priority_q.put((estimated_cost, next_spot, route + [next_spot]))
            
            time.sleep(1)
        
        return None, float('inf')
    
    def smart_astar(self, starting_point, target_point):
        first_route, first_cost = self.find_path_astar(starting_point, target_point)
        
        if first_route:
            best_route = first_route
            best_cost = first_cost
            
            watch_period = 10
            start_watch = time.time()
            
            while time.time() - start_watch < watch_period:
                if self.randomly_adjust_weights():
                    new_route, new_cost = self.find_path_astar(starting_point, target_point, max_attempts=50)
                    
                    if new_route and new_cost < best_cost:
                        best_route = new_route
                        best_cost = new_cost
                
                time.sleep(1)
            
            return best_route, best_cost
        else:
            return None, float('inf')

road_map = {
    'A': {'B': 4, 'C': 3},
    'B': {'E': 12, 'F': 5},
    'C': {'D': 7, 'E': 10},
    'D': {'E': 2},
    'E': {'G': 5},
    'F': {'G': 16},
    'G': {}
}

default_weights = {
    'A': {'B': 4, 'C': 3},
    'B': {'E': 12, 'F': 5},
    'C': {'D': 7, 'E': 10},
    'D': {'E': 2},
    'E': {'G': 5},
    'F': {'G': 16},
    'G': {}
}

guess_values = {'A': 14, 'B': 12, 'C': 11, 'D': 6, 'E': 4, 'F': 11, 'G': 0}

solver = DynamicAStar(road_map, guess_values, default_weights)

found_path, total_expense = solver.smart_astar('A', 'G')

if found_path:
    print(f"Got a route: {' -> '.join(found_path)}")
    print(f"Spent: {total_expense}")
print()

Got a route: A -> C -> E -> G
Spent: 16.1



In [8]:
from queue import PriorityQueue

class DropOffSpot:
    def __init__(self, identifier, coordinates, available_from, available_until, importance=1):
        self.identifier = identifier
        self.coordinates = coordinates
        self.available_from = available_from
        self.available_until = available_until
        self.importance = importance
        self.checked = False
    
    def __str__(self):
        return f"Drop {self.identifier} at {self.coordinates} [{self.available_from}-{self.available_until}]"

def calculate_distance(spot1, spot2):
    return abs(spot1[0] - spot2[0]) + abs(spot1[1] - spot2[1])

class RoutePlanner:
    def __init__(self, warehouse_location):
        self.warehouse = warehouse_location
        self.dropoffs = []
        self.clock = 0
        self.total_travel = 0
        self.path = [warehouse_location]
        
    def add_dropoff(self, dropoff_point):
        self.dropoffs.append(dropoff_point)
    
    def compute_priority_score(self, dropoff, current_spot, current_clock):
        travel_needed = calculate_distance(current_spot, dropoff.coordinates)
        travel_duration = travel_needed
        remaining_slots = dropoff.available_until - current_clock
        
        if remaining_slots <= 0:
            score = 1000 + (100 - dropoff.importance * 10)
        elif remaining_slots < travel_duration:
            score = 500 + (travel_duration - remaining_slots) * 10
        else:
            flexibility = remaining_slots - travel_duration
            score = (dropoff.importance * 50) - (flexibility * 2)
        
        return score
    
    def quick_pick_with_deadlines(self):
        current_spot = self.warehouse
        current_clock = 0
        path = [self.warehouse]
        pending = self.dropoffs.copy()
        total_travel = 0
        completed = []
        
        while pending:
            options = []
            for dropoff in pending:
                priority = self.compute_priority_score(dropoff, current_spot, current_clock)
                travel = calculate_distance(current_spot, dropoff.coordinates)
                options.append((priority, travel, dropoff))
            
            options.sort(reverse=True)
            priority, travel, next_stop = options[0]
            
            arrival = current_clock + travel
            
            if arrival < next_stop.available_from:
                arrival = next_stop.available_from
            
            path.append(next_stop.coordinates)
            current_spot = next_stop.coordinates
            current_clock = arrival + 1
            total_travel += travel
            completed.append(next_stop)
            pending.remove(next_stop)
        
        return_trip = calculate_distance(current_spot, self.warehouse)
        total_travel += return_trip
        path.append(self.warehouse)
        
        return path, total_travel, completed

planner = RoutePlanner((0, 0))

planner.add_dropoff(DropOffSpot('A', (2, 3), 10, 20, importance=3))
planner.add_dropoff(DropOffSpot('B', (5, 1), 5, 15, importance=2))
planner.add_dropoff(DropOffSpot('C', (3, 5), 25, 35, importance=1))
planner.add_dropoff(DropOffSpot('D', (7, 2), 15, 25, importance=3))
planner.add_dropoff(DropOffSpot('E', (4, 7), 30, 45, importance=2))
planner.add_dropoff(DropOffSpot('F', (6, 4), 20, 30, importance=1))

print(f"Starting point: {planner.warehouse}")
print("Places to visit:")
for d in planner.dropoffs:
    print(f"  {d}")

final_route, total_distance, order = planner.quick_pick_with_deadlines()

print(f"\nBest sequence: {final_route}")
print(f"Total travel: {total_distance} units")
print(f"Visit order: {[d.identifier for d in order]}")
print()

Starting point: (0, 0)
Places to visit:
  Drop A at (2, 3) [10-20]
  Drop B at (5, 1) [5-15]
  Drop C at (3, 5) [25-35]
  Drop D at (7, 2) [15-25]
  Drop E at (4, 7) [30-45]
  Drop F at (6, 4) [20-30]

Best sequence: [(0, 0), (2, 3), (5, 1), (7, 2), (4, 7), (6, 4), (3, 5), (0, 0)]
Total travel: 38 units
Visit order: ['A', 'B', 'D', 'E', 'F', 'C']

